# 🧠 05. 프롬프트 조립 & 계층형 메모리 하네스 (Semantic & Episodic Memory)

본 실습 노트북은 프로덕션 에이전트의 핵심 구성 요소인 **`app/middleware/prompt/`**와 **`app/middleware/memory/`**의 아키텍처를 단계별로 직접 실행하며 학습하는 고급 수업용 교재입니다.

---

### 💡 핵심 아키텍처 3가지

1. **최신 에이전트의 표준 5-Layer Prompt Stack & Prompt Caching**
   - **정적 영역 (L1 System Identity + L2 Capabilities [2.1 Tools / 2.2 Skills])**: 모든 세션에서 불변하는 텍스트로 항상 상단에 고정되어 GPU KV-Cache를 보존합니다.
   - **⚡ Boundary Marker (`__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__`)**: Anthropic 캐시 마커 지점.
   - **동적 영역 (L3 Session Context + L4 Memory Documents + L5 Project Rules)**: 매 턴/세션마다 가변되는 영역을 분리 관리합니다.

2. **Semantic Memory (장기 기억: `MEMORY.md` & `USER.md`)**
   - **`§` (Section Sign) 구분자**: 지식/프로필 항목별로 독립된 엔트리를 구성하여 세밀한 수정(`replace`)과 삭제(`remove`)를 가능하게 합니다.
   - **Frozen Snapshot 패턴**: 세션 시작 시 메모리 스냅샷을 시스템 프롬프트에 주입하고 내용을 고정하여 세션 내내 KV-Cache를 보존합니다.

3. **Episodic Memory (세션 기억: SQLite + FTS5) & 2-Stage JIT 회상**
   - **1단계 (Proactive Hint)**: `before_agent`에서 유저 질문을 FTS5로 검색하여 과거 세션 요약(100~200자)만 L4에 선제 주입합니다.
   - **2단계 (On-Demand JIT Recall)**: 에이전트가 상세 맥락이 필요하다고 판단하면 `session_recall` 도구를 스스로 호출하여 원본 메시지를 Anchor 기반으로 인출합니다.

---

### 🎓 학습 목차 (Curriculum Flow)

| 파트 | 주제 | 세부 실행 셀 |
|:---:|:---|:---|
| **Step 0** | **환경 세팅 & 격리 샌드박스** | 루트 탐색, `.env` 로드, `nest_asyncio`, 실습 격리 샌드박스(`demo_dir`) 생성 |
| **Part 1** | **5-Layer Prompt Stack** | 1-1) 도구 및 스킬 카탈로그 빌더 준비<br>1-2) 5-Layer 전체 시스템 프롬프트 조립 & 덤프 출력 |
| **Part 2** | **Semantic Memory (장기 기억) 실습** | 2-1) `§` 엔트리별 프로필 조회<br>2-2) 새 엔트리 추가 & 중복 차단<br>2-3) 특정 엔트리 단위 교체(`replace`)<br>2-4) 시스템 프롬프트 스냅샷 포맷팅 |
| **Part 3** | **Episodic Memory (세션 기억) 실습** | 3-1) 과거 세션 메시지 저장 & FTS5 인덱싱<br>3-2) [비교 실습] LLM 미사용 vs LLM 사용 요약/검색 비교<br>3-3) 유저 질문 기반 1단계 요약 힌트 검색<br>3-4) Anchor 기반 2단계 대화 원문 정밀 인출 |
| **Part 4** | **MemoryMiddleware와 L4 주입 검증** | 4-1) 고응집 도구 목록(`memory`, `session_recall`) 확인<br>4-2) `before_agent` 훅의 L4 주입 실물 덤프 검증 |
| **Part 5** | **`create_agent`로 에이전트 직접 조립 & 시나리오 테스트** | 5-1) 모든 빌딩 블록을 결합하여 에이전트 직접 생성<br>5-2) [시나리오 1] Semantic Memory 개인화 프로필 QA<br>5-3) [시나리오 2] Episodic 8개 대화 중 2-Stage JIT 회상 (`session_recall`)<br>5-4) [시나리오 3] 에이전트 자율 장기 기억 쓰기 (`memory`) |
| **Part 6** | **[학습 정리] 프로덕션 운영 FAQ** | 백그라운드 비동기 요약 주기, 캐시 효율 극대화 팁 |
| **Part 7** | **🧹 Clean-up & Reset (초기화)** | 샌드박스 임시 디렉토리 및 DB 커넥션 완전 정리 |

---

## 🛠️ Step 0. 환경 세팅 & 격리 샌드박스(Sandbox) 초기화

주피터 환경에서 비동기 루프를 안전하게 실행하기 위해 `nest_asyncio`를 활성화하고, **실습 전용 독립 임시 디렉토리(`demo_dir`)**를 생성합니다.

**`§` (Section Sign) 구분자로 명확히 나뉜 영문 마이크로 엔트리 구조**로 `USER.md`와 `MEMORY.md`를 직접 생성하여 독립적인 실습 환경을 구성합니다.

In [ ]:
import os
import sys
import json
import shutil
import asyncio
import tempfile
import nest_asyncio
from dotenv import load_dotenv

# 1. 비동기 루프 패치 (Jupyter 환경 필수)
nest_asyncio.apply()

# 2. 프로젝트 루트 상향 동적 탐색 (어느 서브 폴더에 노트북이 있어도 100% 작동)
def find_project_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "app")) and (
            os.path.exists(os.path.join(p, ".env")) or os.path.exists(os.path.join(p, "configs"))
        ):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.join(os.getcwd(), ".."))

project_root = find_project_root()
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# 3. 환경 변수 로드
dotenv_path = os.path.join(project_root, ".env")
load_dotenv(dotenv_path)

# 4. 실습 격리용 샌드박스 디렉토리 생성
demo_dir = tempfile.mkdtemp(prefix="agent_memory_lab_")
demo_mem_dir = os.path.join(demo_dir, "database")
demo_epi_path = os.path.join(demo_dir, "episodic.db")
demo_chk_path = os.path.join(demo_dir, "checkpoints.db")
os.makedirs(demo_mem_dir, exist_ok=True)

# 5. § 구분자 기반 표준 Semantic Memory 템플릿 생성 (영문)
demo_user_content = """- **Identity**: Cheolsu Kim (AI Software Engineer, 5 years experience)
§
- **Specialty**: AI Agent Engineering, LangChain, LangGraph, Distributed Software Architecture
§
- **Communication Style**: Professional and friendly; strongly prefers explanations focused on core code snippets and architectural diagrams
§
- **Hobbies & Interests**: Golf, weight training, cycling, swimming
§
- **Development Environment**: Primary IDE is VS Code; always provide technical explanations in Korean"""

demo_memory_content = """- **System Architecture**: Claude Code standard 5-layer prompt stack + Hermes multi-layered memory (Semantic & Episodic) harness
§
- **Runtime Environment**: Python 3.12 (WSL Ubuntu) + FastAPI backend + Chainlit UI
§
- **Code Quality Policy**: Always verify generated code snippets for linting and execution validity before presenting them to the user"""

with open(os.path.join(demo_mem_dir, "USER.md"), "w", encoding="utf-8") as f:
    f.write(demo_user_content.strip() + "\n")

with open(os.path.join(demo_mem_dir, "MEMORY.md"), "w", encoding="utf-8") as f:
    f.write(demo_memory_content.strip() + "\n")

# 6. 노트북 하위 잔여 폴더 자동 정리
current_dir = os.getcwd()
for stale_dir in ["app", "artifacts"]:
    stale_path = os.path.join(current_dir, stale_dir)
    if os.path.exists(stale_path):
        shutil.rmtree(stale_path, ignore_errors=True)

print(f"✅ 환경 설정 및 격리 샌드박스 생성 완료!")
print(f"  - Project Root : {project_root}")
print(f"  - 실습용 Sandbox: {demo_dir}")
print(f"  - § 구분자 엔트리 생성: USER.md (5 entries), MEMORY.md (3 entries)")

## 📐 Part 1. Claude Code 5-Layer Prompt Stack 조립 실습

Claude Code의 프롬프트 계층 구조는 **Prompt Caching(KV-Cache)** 효율을 극대화하기 위해 설계되었습니다.

```
┌─────────────────────────────────────────────────────────────────┐
│ Layer 1: System Identity & Core Role (PROMPT.md)                │  <-- [STATIC PREFIX: GPU KV-Cache HIT 🎯]
│ Layer 2: Capabilities (Tools & Skills)                          │
│   ├── 🛠️ Layer 2.1: Tool Capabilities (알파벳 정렬)             │
│   └── 📦 Layer 2.2: Available Skills Catalog (Frontmatter 스캔) │
│ ⚡ __SYSTEM_PROMPT_DYNAMIC_BOUNDARY__ (Cache Control 마킹 지점) │
├─────────────────────────────────────────────────────────────────┤
│ Layer 3: Dynamic Session Context (CWD, Session ID 등)           │  <-- [DYNAMIC SUFFIX: 가변 영역 ⚡]
│ Layer 4: Memory & Dynamic Docs (USER.md / MEMORY.md 전문)        │
│ Layer 5: User & Local Project Rules (AGENT.md)                  │
└─────────────────────────────────────────────────────────────────┘
```

### 1-1. 도구 및 스킬 카탈로그 빌더 준비

In [ ]:
from app.middleware.prompt import PromptAssembler, SkillPromptBuilder
from app.middleware.memory import SemanticMemoryStore, EpisodicStore, MemoryMiddleware
from app.tools import tools_chatbot

# 1. 실제 프로덕션 도구 로드 (기본 챗봇 도구 10개 + 메모리 미들웨어 도구 2개 = 총 12개)
epi_store = EpisodicStore(db_path=demo_epi_path)
await epi_store.setup()
sem_store = SemanticMemoryStore(memory_dir=demo_mem_dir)
sem_store.load_from_disk()

memory_mw = MemoryMiddleware(semantic_store=sem_store, episodic_store=epi_store)
all_real_tools = list(tools_chatbot) + memory_mw.get_tools()
print(f"🛠️ 등록된 실제 프로덕션 도구 수: {len(all_real_tools)}개")
print(f"  - 도구 목록: {[t.name for t in all_real_tools]}")

# 2. 스킬 디렉토리 스캔 및 Frontmatter 카탈로그 빌더 구성
skill_builder = SkillPromptBuilder(
    skills_dirs=[
        os.path.join(project_root, "skills"),
        os.path.join(project_root, ".agents/skills"),
    ],
    guidelines_path=os.path.join(project_root, "app/prompts/SKILL.md")
)
print(f"📦 SkillPromptBuilder 준비 완료")

### 1-2. 5-Layer 시스템 프롬프트 조립 & 덤프 출력

In [ ]:
# 1. Semantic Memory 스냅샷 전문 준비
recalled_memory_parts = []
mem_block = sem_store.format_for_prompt("memory")
usr_block = sem_store.format_for_prompt("user")
if mem_block:
    recalled_memory_parts.append(mem_block)
if usr_block:
    recalled_memory_parts.append(usr_block)
real_recalled_memory = "\n\n".join(recalled_memory_parts)

# 2. 5계층 프롬프트 조립기 초기화
assembler = PromptAssembler(
    system_rules="You are an advanced AI Software Engineer equipped with long-term memory and precise reasoning capabilities.",
    tool_schemas=all_real_tools,  # 실제 12개 도구 객체 주입
    skill_catalog=skill_builder.assemble,  # Callable을 전달하여 동적 Frontmatter 카탈로그 생성
    l5_docs={"AGENT.md": "Always provide clear, structured, and verified responses."}
)

# 3. 세션 컨텍스트 (가변 데이터)
session_context = {
    "cwd": project_root,
    "session_id": "demo_lecture_session_001",
    "os": os.name,
    "user_permission": "ADMIN",
    "active_project": "agent_lab",
    "recalled_memory": real_recalled_memory
}

# 4. 5계층 완성 프롬프트 출력
full_prompt = assembler.build_system_prompt(session_context)
print("=" * 75)
print("📜 [5-Layer 조립된 시스템 프롬프트 출력 (L1 ~ L5 전문)]")
print("=" * 75)
print(full_prompt)
print("=" * 75)

await epi_store.close()

## 💾 Part 2. Semantic Memory (장기 기억: `MEMORY.md` & `USER.md`) 실습

Semantic Memory는 **에이전트의 관찰 노트(`MEMORY.md`)** 와 **사용자 프로필(`USER.md`)** 을 관리합니다.

### 🌟 핵심 설계 원칙
1. **`§` (Section Sign) 구분자**: 각 속성(Identity, Specialty, Hobbies 등)이 독립된 엔트리로 저장되어 세밀한 부분 수정이 가능합니다.
2. **글자 수(Character Count) 용량 제한**: 무한정 메모리가 증가하여 컨텍스트 윈도우를 고갈시키는 문제를 방지합니다.
3. **엔트리 단위 교체 (`replace`)**: `replace(target, old_text, new_content)`는 `old_text`를 포함하는 **해당 엔트리만 찾아 완성된 `new_content`로 독립 교체**합니다.
4. **Frozen Snapshot 패턴**: `load_from_disk()` 시점에 캡처된 스냅샷은 세션 중 도구로 엔트리를 추가해도 **현재 세션의 시스템 프롬프트에는 불변**으로 유지됩니다. (다음 세션부터 반영 ➔ 캐시 보존)

### 2-1. `§` 구분자로 분리된 초기 프로필 엔트리 확인

In [ ]:
store = SemanticMemoryStore(memory_dir=demo_mem_dir, memory_char_limit=4000, user_char_limit=2000)
store.load_from_disk()

print(f"📁 [초기 로드된 사용자 프로필 엔트리: 총 {len(store.user_entries)}개]:")
for idx, entry in enumerate(store.user_entries, 1):
    print(f"  [{idx}] {entry}")

### 2-2. 새 엔트리 추가(`add`) 및 중복 방지 테스트

메모리에 이미 있는 내용은 추가해도 저장되지 않음.

In [ ]:
# 1. 새 엔트리 추가 (새로운 독립 속성 등록)
r_add = store.add("user", "- **Preferred Diagrams**: Mermaid sequence diagrams and ASCII architecture diagrams")
print(f"➕ [추가 결과]: {r_add['message']} (현재 총 {r_add['entry_count']}개 엔트리, 사용량: {r_add['usage']})")

# 2. 동일 엔트리 중복 추가 시도
r_dup = store.add("user", "- **Preferred Diagrams**: Mermaid sequence diagrams and ASCII architecture diagrams")
print(f"🚫 [중복 추가 방지 결과]: {r_dup['message']}")

### 2-3. 특정 엔트리 단위 교체 (`replace`)

`§`로 각 속성이 분리되어 있으므로, **다른 인적사항(Identity, Specialty)에는 전혀 영향을 주지 않고 'Hobbies' 엔트리만 정확하게 수정**할 수 있습니다.

In [ ]:
# 'Hobbies' 키워드로 기존 취미 엔트리만 찾아 완성된 새 문장으로 교체
r_rep = store.replace(
    "user",
    "Hobbies",
    "- **Hobbies & Interests**: Tennis, CrossFit, Road biking, Triathlon"
)
print(f"✏️ [수정 결과]: {r_rep['message']}")
print("\n📁 [수정 후 전체 엔트리 목록]:")
for idx, entry in enumerate(store.user_entries, 1):
    print(f"  [{idx}] {entry}")

### 2-4. 시스템 프롬프트용 스냅샷 렌더링 확인

Part 1의 Prompt assembler에 이와 같은 USER, MEMORY 파일의 내용을 주입하는 기능이 들어있습니다.

In [ ]:
store.load_from_disk()  # 스냅샷 갱신
prompt_block = store.format_for_prompt("user")
print("📄 [시스템 프롬프트 주입용 블록 렌더링 결과 전문]:")
print("─" * 60)
print(prompt_block)
print("─" * 60)

## 📚 Part 3. Episodic Memory (세션 기억: SQLite + FTS5) & 2-Stage JIT 회상 실습

에피소드 메모리는 과거 세션의 대화를 SQLite에 저장하고, **FTS5(Full-Text Search)**를 통해 의미 있는 과거 대화를 검색합니다.

### 🌟 2-Stage JIT (Just-In-Time) 회상 흐름
1. **`finalize_session()`**: 세션 종료 시 메시지 저장 + 요약(Summary) & 키워드 생성 + FTS5 인덱싱.
2. **1단계 (자동 힌트 주입)**: `search_sessions(query)`로 과거 세션 요약 힌트만 L4에 100~200자 주입.
3. **2단계 (Anchor 기반 정밀 인출)**: `get_anchored_view(session_id, anchor_keyword)`로 특정 메시지 중심 ±window 범위 및 첫/끝 bookend 메시지 인출.

### 3-1. 과거 세션 메시지 저장 & FTS5 인덱싱

세션 메시지들을 인덱싱하는 과정을 보여줍니다. 일반적으로 세션이 종료됐다고 판단될 때 대화 내용을 요약해서 인덱싱합니다.

In [ ]:
es = EpisodicStore(db_path=demo_epi_path)
await es.setup()

session_a_msgs = [
    {"role": "human", "content": "도커 컨테이너 빌드 시 캐시 최적화는 어떻게 하나요?"},
    {"role": "ai", "content": "requirements.txt를 먼저 COPY하여 pip install을 수행하고, 소스코드는 나중에 COPY합니다."},
    {"role": "human", "content": "멀티스테이지 빌드는 어떻게 작성하죠?"},
    {"role": "ai", "content": "builder 스테이지에서 빌드 아티팩트를 생성하고, 최종 runner 이미지에는 빌드 도구 없이 최소 런타임만 포함합니다."},
    {"role": "human", "content": "도커 컴포즈에서 restart policy 추천은?"},
    {"role": "ai", "content": "프로덕션 서비스는 restart: always 또는 unless-stopped 설정을 권장합니다."}
]
await es.finalize_session("session_docker_opt_001", session_a_msgs, llm=None)
print("💾 [1. 세션 A 대화 6개 메시지 저장 및 FTS5 인덱싱 완료]")

### 3-2. pisodic session 정보 인덱싱 아키텍처: 1단계(로컬) => 2단계(LLM 사용)

1. **1단계 (로컬 규칙 기반)**: 대화 원문에 실제로 등장한 한/영 단어를 불용어 필터링 후 추출 → FTS5 색인
2. **2단계 (LLM 지능형 보완)**: LLM이 대화 전체를 읽고 핵심 결론 요약 + 원문에 없는 상위 개념·동의어·한영 매핑 키워드 추출 → 1단계와 병합

**💡 핵심 설계 원칙 — 원문 단어 손실 0%:**
- LLM이 키워드를 영어로만 번역하거나, 유의어로 치환해 버려도, **1단계에서 유저가 실제로 말한 한글 원문 단어("쿠키", "보안", "토큰" 등)가 키워드 세트에 100% 보존**됩니다.
- 따라서 한국어로 검색해도, 영어로 검색해도 **검색 누락이 차단**됩니다!

**⚡ 프로덕션 백그라운드 실행:**
- 실제 프로덕션에서는 `MemoryMiddleware.after_agent`가 **백그라운드 데몬 스레드(`daemon=True`)**에서 `finalize_session(llm=review_llm)`을 호출하므로, 사용자 응답 속도에 0ms 지연이 발생합니다.

In [ ]:
from app.utils import init_chat_model

# 1. 요약 테스트용 LLM 초기화 (configs/model.config)
with open(os.path.join(project_root, "configs/model.config"), "r", encoding="utf-8") as f:
    model_cfg = json.load(f)
summary_llm = init_chat_model(
    model=model_cfg.get("model_name", "gemini-3.7-flash"),
    temperature=0.0
)

# 2. 대화 후반부에 핵심 기술 결정(쿠키 보안)이 나오는 대화 데이터셋
compare_msgs = [
    {"role": "human", "content": "오늘 마이크로서비스 인증 아키텍처 회의를 시작합시다."},
    {"role": "ai", "content": "네, 세션 기반 인증과 JWT 토큰 인증 중 어떤 방식을 검토할까요?"},
    {"role": "human", "content": "확장성을 위해 Stateless JWT 방식으로 진행합시다."},
    {"role": "ai", "content": "좋습니다. Stateless JWT 아키텍처로 진행하겠습니다."},
    {"role": "human", "content": "Refresh Token 보안 저장소 위치는 어디로 확정할까요?"},
    {"role": "ai", "content": "XSS 공격 방지를 위해 클라이언트 로컬스토리지 대신 HTTP-Only Secure Cookie에 보관합니다."}
]

# A. LLM 미사용 (1단계 로컬 키워드 추출만 적용)
await es.finalize_session("session_compare_rule_based", compare_msgs, llm=None)

# B. LLM 사용 (1단계 로컬 + 2단계 LLM 지능형 키워드 병합)
await es.finalize_session("session_compare_llm_based", compare_msgs, llm=summary_llm)

# 3. DB에 저장된 실제 요약(Summary) 및 키워드(Keywords) 1:1 대조 출력
rows = await es._conn.execute_fetchall(
    "SELECT session_id, summary, keywords FROM sessions WHERE session_id IN (?, ?)",
    ("session_compare_rule_based", "session_compare_llm_based")
)
print("=" * 75)
print("📊 [1. Progressive Keyword Pipeline 결과 비교]")
print("=" * 75)
for sid, s_summary, s_keywords in rows:
    if sid == "session_compare_rule_based":
        print(f"\n❌ [LLM 미사용 (1단계: 로컬 규칙 기반 키워드만 적용)]:")
        print(f"   - Summary : {s_summary}")
        print(f"   - Keywords: {s_keywords}")
    elif sid == "session_compare_llm_based":
        print(f"\n🎯 [LLM 사용 (1단계 로컬 + 2단계 LLM 한/영 키워드 병합)]:")
        print(f"   - Summary : {s_summary}")
        print(f"   - Keywords: {s_keywords}")

In [ ]:
# 4. 핵심 비교: '쿠키 보안' 한국어 FTS5 검색 결과
print("\n" + "=" * 75)
print("🔍 [2. '쿠키 보안' 한국어 검색 — 원문 키워드 보존 효과 검증]")
print("=" * 75)
search_cookie = await es.search_sessions("쿠키 보안", top_k=5)
cookie_sids = [(s["session_id"], s['summary']) for s in search_cookie]

print(f"🔎 한국어 검색어: '쿠키 보안' ➔ 검색된 세션: {cookie_sids}")

if not cookie_sids:
    print("  ⚠️ 검색 결과 없음 — 키워드 인덱스를 확인하세요.")

In [ ]:
# 5. 영문 기술어 검색 — LLM 보완의 부가가치 확인
print("\n" + "=" * 75)
print("🔍 [3. 'Stateless Authentication' 영문 기술어 검색 — LLM 보완 효과 검증]")
print("=" * 75)
search_auth = await es.search_sessions("Stateless Authentication", top_k=5)
auth_sids = [(s["session_id"], s['summary']) for s in search_auth]

print(auth_sids)

### 3-3. 유저 질문 기반 FTS5 검색 (`search_sessions`)

관련 세션을 검색해보는 1차 검색으로 차후 프롬프트에 Epidodic Memory로 자동 주입될 정보. 이 정보는 관련한 대화에 대한 기억을 에이전트가 자동으로 떠올리는 것에 비유할 수 있음.

In [ ]:
search_results = await es.search_sessions("도커 멀티스테이지 빌드", top_k=2)
print("🔍 [FTS5 검색 결과 (1단계 검색)]:")
for res in search_results:
    print(f"  - Session ID: {res['session_id']}")
    print(f"    Summary   : {res['summary']}")
    print(f"    Keywords  : {res['keywords']}")

### 3-4. Anchor 기반 2단계 대화 인출 (`session_recall`)

Session ID와 keyword로 관련한 대화 원문을 추출합니다. 에이전트의 도구를 통해 주입될 정보. 즉 에이전트가 1차 검색을 토대로 필요하다고 생각할 때, 기억을 더듬어 관련 대화 기록을 불러오는 기능.

In [ ]:
anchored_msgs = await es.get_anchored_view(
    session_id="session_docker_opt_001",
    anchor_keyword="멀티스테이지",
    window=2
)
print(f"🎯 [Anchor('멀티스테이지') 기반 정밀 인출 ({len(anchored_msgs)}개 메시지 전문)]:")
for idx, m in enumerate(anchored_msgs, 1):
    print(f"  [{idx}] {m['role'].upper()}: {m['content']}")

await es.close()

## 🔌 Part 4. MemoryMiddleware와 고응집 도구 바인딩 & L4 주입 검증

미들웨어가 메모리 스토어뿐 아니라 에이전트가 사용할 **도구(`memory`, `session_recall`)까지 소유**하는 고응집(High-Cohesion) 구조입니다.

### 🌟 `before_agent` 훅을 통한 L4 메모리 단일 주입 과정
1. `semantic_store.format_for_prompt()` ➔ **Semantic Frozen Snapshot**
2. `episodic_store.search_sessions(user_query)` ➔ **Episodic FTS5 요약 힌트**
3. 위 두 가지를 `ctx.recalled_memory`에 결합하여 L4에 단일 주입 (이중 주입 방지)

### 4-1. 메모리 도구 목록 확인

Memory Middleware를 통해 자동 주입되는 메모리와 관련한 도구들.

In [ ]:
sem_store = SemanticMemoryStore(memory_dir=demo_mem_dir)
sem_store.load_from_disk()
epi_store = EpisodicStore(db_path=demo_epi_path)
await epi_store.setup()

mw = MemoryMiddleware(semantic_store=sem_store, episodic_store=epi_store)

tools = mw.get_tools()
print(f"🛠️ MemoryMiddleware가 제공하는 도구 목록 ({len(tools)}개):")
for t in tools:
    print(f"  - 이름: {t.name:<15} 설명: {t.description}...")

### 4-2. `before_agent` 훅 실행 및 L4 주입 데이터 덤프 검증

Memory Middleware는 before_agent 단계에서 runtime의 recalled_memory 변수에 기억 주입. 차후 Prompt assembler Middleware에 의해 프롬프트에 주입.

In [ ]:
from langchain_core.messages import HumanMessage
from app.utils.context import AgentContext

ctx = AgentContext(semantic_memory_enabled=True, episodic_memory_enabled=True)
class DummyRuntime:
    def __init__(self, ctx):
        self.context = ctx
        self.config = {"configurable": {"thread_id": "demo_thread_001"}}

fake_state = {"messages": [HumanMessage(content="도커 빌드 캐시 최적화에 대해 질문할게요.")]}
mw.before_agent(fake_state, DummyRuntime(ctx))

print("📥 [before_agent 훅에 의해 ctx.recalled_memory에 자동 주입된 전체 내용 (전문)]:")
print("─" * 75)
print(ctx.recalled_memory)
print("─" * 75)

await epi_store.close()

## 🚀 Part 5. `create_agent`로 에이전트 직접 조립 및 시나리오 테스트

Part 1~4에서 살펴본 **LLM, 5-Layer PromptAssembler, MemoryMiddleware, Checkpointer**를 `create_agent`를 통해 **노트북 내에서 직접 조립**하고, 3가지 시나리오를 각각 독립 셀에서 실행합니다.

### 5-1. 모든 빌딩 블록을 결합하여 `create_agent`로 직접 에이전트 인스턴스 빌드

In [ ]:
import aiosqlite
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from app.utils import init_chat_model
from app.middleware.prompt import PromptAssembler, SkillPromptBuilder, create_prompt_assembler_middleware
from app.middleware.memory import SemanticMemoryStore, EpisodicStore, MemoryMiddleware
from app.tools import tools_chatbot
from app.utils.context import AgentContext
from app.utils.message_utils import normalize_content

print("🤖 [Memory Agent 직접 조립 중...]")

# 1. LLM 초기화 (configs/model.config 파일 로드)
with open(os.path.join(project_root, "configs/model.config"), "r", encoding="utf-8") as f:
    model_cfg = json.load(f)
llm = init_chat_model(
    model=model_cfg.get("model_name", "gemini-3.7-flash"),
    temperature=model_cfg.get("temperature", 0.0)
)

# 2. 스토어 및 미들웨어 초기화 (실습 샌드박스 경로 연결)
sem_store = SemanticMemoryStore(memory_dir=demo_mem_dir)
sem_store.load_from_disk()

epi_store = EpisodicStore(db_path=demo_epi_path)
await epi_store.setup()

memory_mw = MemoryMiddleware(semantic_store=sem_store, episodic_store=epi_store, review_llm=llm)
all_tools = list(tools_chatbot) + memory_mw.get_tools()

# 3. 5-Layer Prompt Middleware 구성
skill_builder = SkillPromptBuilder(
    skills_dirs=[os.path.join(project_root, "skills"), os.path.join(project_root, ".agents/skills")],
    guidelines_path=os.path.join(project_root, "app/prompts/SKILL.md") if os.path.exists(os.path.join(project_root, "app/prompts/SKILL.md")) else None,
)
assembler = PromptAssembler(
    system_rules="You are an advanced AI Software Engineer equipped with long-term memory and precise reasoning capabilities.",
    tool_schemas=all_tools,
    skill_catalog=skill_builder.assemble,
    l4_docs={},
    agent_rules_path=os.path.join(project_root, "app/prompts/SKILL.md") if os.path.exists(os.path.join(project_root, "app/prompts/SKILL.md")) else None,
)
prompt_mw = create_prompt_assembler_middleware(assembler, merge_system=True)

# 4. Checkpointer 연결 (샌드박스 DB)
chk_conn = await aiosqlite.connect(demo_chk_path, check_same_thread=False)
checkpointer = AsyncSqliteSaver(chk_conn)
await checkpointer.setup()

# 5. create_agent로 에이전트 빌드!
agent = create_agent(
    model=llm,
    tools=all_tools,
    middleware=[memory_mw, prompt_mw],
    checkpointer=checkpointer,
    context_schema=AgentContext,
)

# 참조 저장 (실습 시나리오 및 리소스 정리용)
agent.episodic_store = epi_store
agent.semantic_store = sem_store
agent.checkpointer_conn = chk_conn

ctx = AgentContext(
    semantic_memory_enabled=True,
    episodic_memory_enabled=True,
    user_permission="ADMIN",
    active_project="agent_lab"
)
print("✅ create_agent를 통한 에이전트 조립 및 준비 완료!")

### 5-2. [시나리오 1] Semantic Memory 기반 개인화 프로필 QA (`USER.md` 반영)

In [ ]:
query_1 = "내 이름과 전문 분야, 그리고 내가 선호하는 설명 스타일을 알려줘."
print(f"👤 질문: {query_1}")

res_1 = await agent.ainvoke(
    {"messages": [HumanMessage(content=query_1)]},
    config={"configurable": {"thread_id": "scenario_01"}},
    context=ctx
)
ans_1 = normalize_content([m for m in res_1["messages"] if isinstance(m, AIMessage)][-1].content)
print(f"\n🤖 답변 (전문):\n{ans_1}")

### 5-3. [시나리오 2] Episodic 8개 대화 중 2-Stage JIT 회상 (`session_recall` Window 슬라이싱)

In [ ]:
# 1. 과거 세션 회의록 데이터 8개 메시지 저장 (설계 -> 정책 -> 저장소 논의)
past_sid = "session_jwt_auth_policy_2026"
session_jwt_msgs = [
    {"role": "human", "content": "오늘 마이크로서비스 인증 아키텍처 설계 회의를 시작합시다."},
    {"role": "ai", "content": "네, 세션 기반 인증과 JWT 토큰 인증 중 어떤 방식을 검토할까요?"},
    {"role": "human", "content": "확장성을 위해 Stateless JWT 인증 방식을 채택합시다."},
    {"role": "ai", "content": "좋습니다. Stateless JWT 아키텍처로 진행하겠습니다."},
    {"role": "human", "content": "JWT 토큰의 Access Token과 Refresh Token 유효기간 정책은 어떻게 되나요?"},
    {"role": "ai", "content": "Access Token은 15분, Refresh Token은 7일로 확정했습니다."},
    {"role": "human", "content": "Refresh Token은 보안상 어디에 저장할 계획인가요?"},
    {"role": "ai", "content": "XSS 공격 방지를 위해 클라이언트 저장소가 아닌 HTTP-Only Secure Cookie에 보관합니다."}
]
await agent.episodic_store.finalize_session(past_sid, session_jwt_msgs, llm=None)
print(f"💾 [과거 세션({past_sid}) 회의록 8개 메시지 저장 및 FTS5 인덱싱 완료]")

# 2. 유저 질문: 에이전트는 1단계 선제 요약 힌트를 확인한 뒤, 정밀 인출을 위해 session_recall 도구를 호출합니다.
query_2 = "이전에 우리가 정한 JWT 토큰 만료 정책(Access/Refresh 유효기간)이 뭐였지? session_recall 도구로 확인해줘."
print(f"\n👤 질문: {query_2}")

res_2 = await agent.ainvoke(
    {"messages": [HumanMessage(content=query_2)]},
    config={"configurable": {"thread_id": "scenario_02"}},
    context=ctx
)

# 3. 도구 호출 궤적 분석 (전체 8개 중 관련 Window 및 Bookend 인출 확인)
for m in res_2["messages"]:
    if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
        print(f"\n🎯 [에이전트 도구 호출]: {m.tool_calls[0]['name']}({m.tool_calls[0]['args']})")
    elif isinstance(m, ToolMessage):
        print(f"\n🔧 [도구 반환 데이터 (전체 8개 중 Anchor Window 기반 발췌 전문)]:\n{m.content}")

ans_2 = normalize_content([m for m in res_2["messages"] if isinstance(m, AIMessage)][-1].content)
print(f"\n🤖 답변 (전문):\n{ans_2}")

### 5-4. [시나리오 3] memory 도구를 활용한 장기 기억 자율 쓰기

In [ ]:
query_3 = "다음을 기억해줘: 나의 최애 과일은 애플망고야. memory 도구의 add 액션으로 user 타겟에 저장해."
print(f"👤 지시: {query_3}")

res_3 = await agent.ainvoke(
    {"messages": [HumanMessage(content=query_3)]},
    config={"configurable": {"thread_id": "scenario_03"}},
    context=ctx
)

for m in res_3["messages"]:
    if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
        print(f"  🎯 [에이전트 도구 호출]: {m.tool_calls[0]['name']}({m.tool_calls[0]['args']})")
    elif isinstance(m, ToolMessage):
        print(f"  🔧 [도구 반환 데이터 전문]: {m.content}")

ans_3 = normalize_content([m for m in res_3["messages"] if isinstance(m, AIMessage)][-1].content)
print(f"\n🤖 답변 (전문):\n{ans_3}")

## 🎓 Part 6. [학습 정리] 프로덕션 운영 핵심 FAQ

### Q1. Episodic Memory는 언제 업데이트되나요?
- **답변**: `after_agent` 훅에서 대화가 끝난 직후 **비동기 데몬 스레드(`daemon=True`)**가 백그라운드로 스폰되어 `finalize_session()`을 실행합니다.
- 사용자에게 나가는 최종 응답에는 **0ms의 지연**을 주며, 백그라운드에서 SQLite에 대화 원문을 저장하고 LLM 영어 요약 및 FTS5 인덱싱을 수행합니다.

### Q2. 왜 `session_search`를 도구로 주지 않고 `session_recall`만 주었나요?
- **답변**: **도구 피로(Tool Fatigue) 방지 및 프롬프트 캐싱 최적화** 때문입니다.
- 검색(Search)은 `MemoryMiddleware.before_agent()`가 유저 질문을 바탕으로 자동으로 수행하여 L4에 100자 힌트만 넣어줍니다.
- 에이전트는 이미 요약된 힌트를 보고, 정말 필요한 경우에만 `session_recall` 도구로 원본 메시지를 열람합니다. (2-Stage JIT 패턴)

### Q3. Semantic Memory의 'Frozen Snapshot'이 왜 중요한가요?
- **답변**: 세션 도중에 에이전트가 `memory(add)`로 새 기억을 추가했다고 해서 **현재 세션의 시스템 프롬프트가 매 턴마다 바뀌면 GPU KV-Cache가 모두 깨져버립니다.**
- 따라서 현재 세션에서는 세션 시작 시점의 스냅샷을 고정 사용하고, 디스크에만 즉시 저장해 둔 뒤 **다음 세션부터 최신 마크다운을 로드**합니다.

## 🧹 Part 7. Clean-up & Reset (실습 리소스 초기화)

실습이 모두 완료된 후 생성되었던 **임시 샌드박스 디렉토리(`demo_dir`)와 열려있는 DB 커넥션을 깔끔하게 종료 및 삭제**하여 다음 실습 때도 항상 깨끗한 상태에서 시작할 수 있도록 초기화합니다.

In [ ]:
# 1. 열려있는 DB 커넥션 종료
if 'agent' in locals():
    if hasattr(agent, "checkpointer_conn"):
        await agent.checkpointer_conn.close()
    if hasattr(agent, "episodic_store"):
        await agent.episodic_store.close()

# 2. 실습 임시 샌드박스 디렉토리 완전 삭제
if 'demo_dir' in locals() and os.path.exists(demo_dir):
    shutil.rmtree(demo_dir, ignore_errors=True)
    print(f"🧹 실습 임시 샌드박스 삭제 완료: {demo_dir}")

# 3. 혹시 남아있을 수 있는 노트북 하위 잔여 폴더 정리
for stale in ["app", "artifacts"]:
    p = os.path.join(current_dir, stale)
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)

print("✨ 모든 실습 리소스가 성공적으로 초기화되었습니다! (Ready for next run)")